<a href="https://colab.research.google.com/github/gussgary/Financial-Fraud-Detection/blob/main/Fraud_Analysis_Modelling_and_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Core data manipulation and numerical computing
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Machine Learning -- preprocessing & model selection
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_val_score, GridSearchCV, RandomizedSearchCV, learning_curve
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif

# Machine Learning -- classifiers
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgb

# Imbalanced-learn -- resampling
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek

# Evaluation metrics
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score, accuracy_score,
    precision_recall_fscore_support, balanced_accuracy_score
)

# Clustering (used in V-feature engineering)
from sklearn.cluster import KMeans

# Global settings
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('All libraries loaded successfully.')
print(f'Random seed: {RANDOM_STATE}')

All libraries loaded successfully.
Random seed: 42


In [8]:
import pickle
import gdown
import os

FILES = {
    'X_train.parquet': '1I5JSEoEG_qviwFmRWfSuznNICxoyRJJL',
    'y_train.parquet':'112UMC8JfsLL6BjIv7WrYPzDbSit2h4Tm',
    'X_test.parquet': '1LumEaDKIcH8oRz9q9v66G5cEVYp2FCYT',
    'y_test.parquet': '1mMgOpzWjZQFRCfM33TKFhMUNADQzevvJ'

}

for filename, fileID in FILES.items():
    if not os.path.exists(filename):
        print(f'Downloading {filename}...')
        gdown.download(f'https://drive.google.com/uc?id={fileID}', filename, quiet=False)
    else:
        print(f'{filename} already exists, skipping...')

X_train = pd.read_parquet('X_train.parquet')
y_train = pd.read_parquet('y_train.parquet').squeeze()
X_test  = pd.read_parquet('X_test.parquet')
y_test  = pd.read_parquet('y_test.parquet').squeeze()

print(f'X_train: {X_train.shape}')
print(f'y_train: {y_train.shape}')
print(f'X_test : {X_test.shape}')
print(f'y_test : {y_test.shape}')

Downloading...
From (original): https://drive.google.com/uc?id=1I5JSEoEG_qviwFmRWfSuznNICxoyRJJL
From (redirected): https://drive.google.com/uc?id=1I5JSEoEG_qviwFmRWfSuznNICxoyRJJL&confirm=t&uuid=84150429-c7dd-498e-ab96-17a7a0b4b763
To: /content/X_train.parquet
100%|██████████| 542M/542M [00:13<00:00, 40.9MB/s]


Downloading...
From: https://drive.google.com/uc?id=112UMC8JfsLL6BjIv7WrYPzDbSit2h4Tm
To: /content/y_train.parquet
100%|██████████| 41.0k/41.0k [00:00<00:00, 52.3MB/s]


Downloading...
From: https://drive.google.com/uc?id=1LumEaDKIcH8oRz9q9v66G5cEVYp2FCYT
To: /content/X_test.parquet
100%|██████████| 22.9M/22.9M [00:00<00:00, 28.4MB/s]


Downloading...
From: https://drive.google.com/uc?id=1mMgOpzWjZQFRCfM33TKFhMUNADQzevvJ
To: /content/y_test.parquet
100%|██████████| 10.6k/10.6k [00:00<00:00, 15.4MB/s]


X_train: (911804, 366)
y_train: (911804,)
X_test : (118108, 366)
y_test : (118108,)
